# 학습목표
- Iceberg의 **Time Travel 기능**을 이해하고 과거 시점의 데이터를 조회하는 방법을 학습한다.  
- Snapshot의 **ID, Timestamp, Parent ID, Sequence 구조**를 파악하여 왜 Time Travel이 빠르고 효율적인지 이해한다.  
- **Rollback**을 통해 과거로 되돌아가는 방법과, **Set Current Snapshot**을 통해 앞뒤 시점으로 자유롭게 이동하는 방법을 실습한다.  
- 실무에서 잘못된 데이터 수정이나 삭제를 복구할 때 Time Travel을 활용하는 구체적 시나리오를 익힌다.  


---

## 📌 TL;DR – Iceberg Time Travel

- **Time Travel**은 과거 시점의 데이터를 조회하는 기능으로, 잘못된 수정/삭제를 복구할 때 유용하다.  
- Iceberg는 **Snapshot(스냅샷)** 기반으로 동작하며, 각 Snapshot은 **ID, Timestamp, Parent, Sequence** 정보를 가진다.  
- Time Travel 방법은 두 가지가 있다:  
  - [**Rollback**](https://iceberg.apache.org/docs/1.5.1/spark-procedures/#snapshot-management) → 과거로만 이동 가능 (Snapshot ID 또는 Timestamp 사용).  
  - [**Set Current Snapshot**](https://iceberg.apache.org/docs/1.5.1/spark-procedures/#set_current_snapshot) → 특정 Snapshot을 현재로 설정 가능 (앞뒤 이동 모두 가능).  
- Snapshot 체계를 이해하면 데이터 오류 상황에서도 **빠르고 안전하게 복원**할 수 있다.  


---

In [1]:
import io
import json
import boto3
from fastavro import reader as avro_reader
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from IPython.display import Image
from utils.get_iceberg_metadata import get_metadata, print_avro_item
from utils.iceberg_metadata_properties import IcebergMetadataJsonField, FIELD_DESCRIPTIONS

In [2]:
spark = SparkSession.builder \
    .appName("IcebergSparkWithMinIO") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "rest") \
    .config("spark.sql.catalog.my_catalog.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.my_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.my_catalog.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.my_catalog.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.my_catalog.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.my_catalog.s3.path-style-access", "true") \
    .getOrCreate()

25/10/11 13:22:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS my_catalog.my_database")

DataFrame[]

In [4]:
catalog = "my_catalog"
database = "my_database"
table = "my_timetravel"
table_full_name = f"{catalog}.{database}.{table}"
table_full_name

'my_catalog.my_database.my_timetrevel'

In [5]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_full_name} (
        id BIGINT,
        name STRING
    ) USING iceberg
""")

DataFrame[]

In [6]:
spark.sql(f"SELECT * FROM {table_full_name}").show()

+---+----+
| id|name|
+---+----+
+---+----+



## INSERT(첫 번째 스냅샷)

In [7]:
spark.sql(f"""
    INSERT INTO {table_full_name} VALUES 
    (1, 'Alice'), (2, 'Bob')
""")
spark.sql(f"""
    SELECT * FROM {table_full_name}
""").show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



## INSERT 후 SNAPSHOT 이력 확인
- 첫 데이터 삽입 단계라서 parent_id(이전 스냅샷 id)는 NULL 이다.

In [8]:
spark.sql(f"""
    SELECT * FROM {table_full_name}.snapshots
""").show()
spark.sql(f"""
    SELECT manifest_list FROM {table_full_name}.snapshots
""").show(truncate=False)

+--------------------+-------------------+---------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+---------+---------+--------------------+--------------------+
|2025-10-11 13:23:...|3346042836438594978|     NULL|   append|s3://warehouse/my...|{spark.app.id -> ...|
+--------------------+-------------------+---------+---------+--------------------+--------------------+

+----------------------------------------------------------------------------------------------------------------------+
|manifest_list                                                                                                         |
+----------------------------------------------------------------------------------------------------------------------+
|s3://warehouse/my_database/my_timetrevel/metadata/snap-3346042836438594978-1-eed8bd7a-887d-48a4-8401-80a15f4c1

- snapshot(manifest list) 파일 내용 확인하기

In [10]:
manifest_list = "snap-3346042836438594978-1-eed8bd7a-887d-48a4-8401-80a15f4c1e52.avro"
obj = get_metadata(file_name=manifest_list, table=table)
print_avro_item(obj)

{
  "manifest_path": "s3://warehouse/my_database/my_timetrevel/metadata/eed8bd7a-887d-48a4-8401-80a15f4c1e52-m0.avro",
  "manifest_length": 7050,
  "partition_spec_id": 0,
  "content": 0,
  "sequence_number": 1,
  "min_sequence_number": 1,
  "added_snapshot_id": 3346042836438594978,
  "added_files_count": 2,
  "existing_files_count": 0,
  "deleted_files_count": 0,
  "added_rows_count": 2,
  "existing_rows_count": 0,
  "deleted_rows_count": 0,
  "partitions": [],
  "key_metadata": null
}


## UPDATE(두 번째 스냅샷)

In [11]:
spark.sql(f"""
    UPDATE {table_full_name} VALUES SET name = 'David' WHERE id = 1
""")

+---+-----+
| id| name|
+---+-----+
|  1|David|
|  2|  Bob|
+---+-----+



- 조회 결과와 같이 snapshot 파일(`snapshot_id = 6155482871164859340`)이 하나 더 생겼다.

In [12]:
spark.sql(f"""
    SELECT * FROM {table_full_name}
""").show()
spark.sql(f"""
    SELECT manifest_list FROM {table_full_name}.snapshots
""").show(truncate=False)

+---+-----+
| id| name|
+---+-----+
|  1|David|
|  2|  Bob|
+---+-----+

+----------------------------------------------------------------------------------------------------------------------+
|manifest_list                                                                                                         |
+----------------------------------------------------------------------------------------------------------------------+
|s3://warehouse/my_database/my_timetrevel/metadata/snap-3346042836438594978-1-eed8bd7a-887d-48a4-8401-80a15f4c1e52.avro|
|s3://warehouse/my_database/my_timetrevel/metadata/snap-6155482871164859340-1-3e051ff5-472e-4be8-81b8-305bc8a95758.avro|
+----------------------------------------------------------------------------------------------------------------------+



- UPDATE 후 새로 생긴 snapshot 내용을 확인해보자.
  - sequence_number 가 1 -> 2로 증가하였다.
  - `6155482871164859340` 의 스냅샷의 parent_id 가 이전 스냅샷 ID 로 지정되었다.

In [15]:
spark.sql(f"""
    SELECT snapshot_id, parent_id, operation FROM {table_full_name}.snapshots
""").show(truncate=False)

+-------------------+-------------------+---------+
|snapshot_id        |parent_id          |operation|
+-------------------+-------------------+---------+
|3346042836438594978|NULL               |append   |
|6155482871164859340|3346042836438594978|overwrite|
+-------------------+-------------------+---------+



In [13]:
manifest_list = "snap-6155482871164859340-1-3e051ff5-472e-4be8-81b8-305bc8a95758.avro"
obj = get_metadata(file_name=manifest_list, table=table)
print_avro_item(obj)

{
  "manifest_path": "s3://warehouse/my_database/my_timetrevel/metadata/3e051ff5-472e-4be8-81b8-305bc8a95758-m1.avro",
  "manifest_length": 7026,
  "partition_spec_id": 0,
  "content": 0,
  "sequence_number": 2,
  "min_sequence_number": 2,
  "added_snapshot_id": 6155482871164859340,
  "added_files_count": 1,
  "existing_files_count": 0,
  "deleted_files_count": 0,
  "added_rows_count": 1,
  "existing_rows_count": 0,
  "deleted_rows_count": 0,
  "partitions": [],
  "key_metadata": null
}
{
  "manifest_path": "s3://warehouse/my_database/my_timetrevel/metadata/3e051ff5-472e-4be8-81b8-305bc8a95758-m0.avro",
  "manifest_length": 7062,
  "partition_spec_id": 0,
  "content": 0,
  "sequence_number": 2,
  "min_sequence_number": 1,
  "added_snapshot_id": 6155482871164859340,
  "added_files_count": 0,
  "existing_files_count": 1,
  "deleted_files_count": 1,
  "added_rows_count": 0,
  "existing_rows_count": 1,
  "deleted_rows_count": 1,
  "partitions": [],
  "key_metadata": null
}


## INSERT(세 번째 스냅샷)
- 두 개의 행을 추가로 삽입하여 스냅샷을 추가한다.

In [17]:
spark.sql(f"""
    INSERT INTO {table_full_name} VALUES
    (3, 'Jaemin'), (4, 'Moo')
""")
spark.sql(f"""
    SELECT * FROM {table_full_name}
""").show()

+---+------+
| id|  name|
+---+------+
|  1| David|
|  3|Jaemin|
|  4|   Moo|
|  2|   Bob|
+---+------+



- 아래 조회 결과와 같이 스냅샷 파일이 하나 더 추가 되어 총 3개가 되었다.

In [18]:
spark.sql(f"""
    SELECT manifest_list FROM {table_full_name}.snapshots
""").show(truncate=False)
spark.sql(f"""
    SELECT snapshot_id, parent_id, operation FROM {table_full_name}.snapshots
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------+
|manifest_list                                                                                                         |
+----------------------------------------------------------------------------------------------------------------------+
|s3://warehouse/my_database/my_timetrevel/metadata/snap-3346042836438594978-1-eed8bd7a-887d-48a4-8401-80a15f4c1e52.avro|
|s3://warehouse/my_database/my_timetrevel/metadata/snap-6155482871164859340-1-3e051ff5-472e-4be8-81b8-305bc8a95758.avro|
|s3://warehouse/my_database/my_timetrevel/metadata/snap-3432909901350970807-1-66fd1302-acb4-45e5-adec-327b80ab5dbd.avro|
+----------------------------------------------------------------------------------------------------------------------+

+-------------------+-------------------+---------+
|snapshot_id        |parent_id          |operation|
+-------------------+-------------------+-------

---

## `rollback_to_snapshot` 프로시저 사용해보기
- `rollback_to_snapshot`는 테이블을 특정 스냅샷 ID로 롤백 가능하다.
- 가장 처음 생성된 스냅샷으로 롤백해보자.

In [19]:
spark.sql(f"""
    CALL {catalog}.system.rollback_to_snapshot(
        table => '{database}.{table}',
        snapshot_id => 3346042836438594978
    ) 
""").show()

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 3432909901350970807|3346042836438594978|
+--------------------+-------------------+



- 위 조회 결과에서 `3432909901350970807` 스냅샷에서 `3346042836438594978` 스냅샷으로 변경 된 것을 알 수 있다.
- rollback_to_snapshot 프로시저 사용 후 다시 테이블 데이터를 조회하면 아래와 같이 가장 처음 생성된 스냅샷 시점의 데이터가 조회된다.

In [21]:
spark.sql(f"""
    SELECT * FROM {table_full_name}
""").show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



- `rollback_to_snapshot`는 과거 스냅샷으로만 이동 가능하다고 했다.
- 미래 시점의 스냅샷 ID 를 사용할 경우 오류가 나는지 확인해보자.
- 다음과 같이 오류가 발생한다.
  - `org.apache.iceberg.exceptions.ValidationException: Cannot roll back to snapshot, not an ancestor of the current state: 6155482871164859340`

In [23]:
# current_snapshot_id 보다 미래의 snapshot_id 로 rollback_to_snapshot 를 시도할 경우
spark.sql(f"""
    CALL {catalog}.system.rollback_to_snapshot(
        table => '{database}.{table}',
        snapshot_id => 6155482871164859340
    ) 
""").show()

Py4JJavaError: An error occurred while calling o36.sql.
: org.apache.iceberg.exceptions.ValidationException: Cannot roll back to snapshot, not an ancestor of the current state: 6155482871164859340
	at org.apache.iceberg.exceptions.ValidationException.check(ValidationException.java:49)
	at org.apache.iceberg.SetSnapshotOperation.rollbackTo(SetSnapshotOperation.java:84)
	at org.apache.iceberg.SnapshotManager.rollbackTo(SnapshotManager.java:67)
	at org.apache.iceberg.spark.procedures.RollbackToSnapshotProcedure.lambda$call$0(RollbackToSnapshotProcedure.java:88)
	at org.apache.iceberg.spark.procedures.BaseProcedure.execute(BaseProcedure.java:107)
	at org.apache.iceberg.spark.procedures.BaseProcedure.modifyIcebergTable(BaseProcedure.java:88)
	at org.apache.iceberg.spark.procedures.RollbackToSnapshotProcedure.call(RollbackToSnapshotProcedure.java:83)
	at org.apache.spark.sql.execution.datasources.v2.CallExec.run(CallExec.scala:34)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandEx

---

## `set_current_snapshot` 프로시저 사용해보기
- `set_current_snapshot`는 테이블을 현재 스냅샷 ID를 지정해줄 수 있다.
- 현재는 첫 번째 스냅샷 ID가 활성화 되어 있으므로, 두 번째 미래의 스냅샷 ID(`6155482871164859340`) 로 지정해보자.

In [33]:
spark.sql(f"""
    CALL {catalog}.system.set_current_snapshot(
        table => '{database}.{table}',
        snapshot_id => 6155482871164859340
    )
""").show()

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 3346042836438594978|6155482871164859340|
+--------------------+-------------------+



- 위 출력 결과와 같이 current_snapshot_id 가 두 번째 스냅샷으로 정상 이동 하였다.
- `set_current_snapshot`는 아래 문서와 같이 옵션 파라미터로 `snapshot_id` 또는 `reg(브랜치 또는 태그정보)` 둘 중 하나를 사용할 수 있다.
  - https://iceberg.apache.org/docs/1.5.1/spark-procedures/#set_current_snapshot

---

## `rollback_to_timestamp` 프로시저 사용해보기
- `rollback_to_timestamp`는`timestamp` 를 파리미터로 전달하여 rollback 할 수 있다.
  - **timestamp 파라미터로 지정된 시간 미만의 가장 최신 스냅샷으로 이동한다.**
- 먼저, 각 스냅샷의 timestamp 값(=committed_at 컬럼값)을 확인해보자.

In [34]:
spark.sql(f"""
    SELECT committed_at, snapshot_id FROM {table_full_name}.snapshots
""").show(truncate=False)

+-----------------------+-------------------+
|committed_at           |snapshot_id        |
+-----------------------+-------------------+
|2025-10-11 13:23:06.436|3346042836438594978|
|2025-10-11 13:24:47.331|6155482871164859340|
|2025-10-11 13:33:05.682|3432909901350970807|
+-----------------------+-------------------+



In [36]:
spark.sql(f"""
    CALL {catalog}.system.rollback_to_timestamp(
        table => '{database}.{table}',
        timestamp => TIMESTAMP '2025-10-11 13:23:06.436'
    )
""").show()

IllegalArgumentException: Cannot roll back, no valid snapshot older than: 1760188986436

- `3346042836438594978` 스냅샷으로 롤백하기 위해 해당 스냅샷의 `committed_at`을 timestamp 파라미터로 전달하였으나 아래와 같이 오류가 난다.
- 오류:`IllegalArgumentException: Cannot roll back, no valid snapshot older than: 1760188986436`
- 이유: 이유는 timestamp로 지정한 시각보다 **이전(strictly older)인 스냅샷**을 ㅡ선택하기 떄문이다.
- 대응: timestamp 파라미터에 1초를 더하여 다시 명령해보자.

In [39]:
spark.sql(f"""
    CALL {catalog}.system.rollback_to_timestamp(
        table => '{database}.{table}',
        timestamp => TIMESTAMP '2025-10-11 13:23:07.436'
    )
""").show()

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 6155482871164859340|3346042836438594978|
+--------------------+-------------------+



- 위 결과와 같이 정상적으로 `current_snapshot_id` 가 첫 번째 스냅샷 시점으로 변경 되었다.
- 그리고 아래와 같이 첫 번째 스냅샷 시점의 데이터가 잘 조회된다.

In [42]:
spark.sql(f"""
    SELECT * FROM {table_full_name}
""").show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

